# VMC2026 Track 3 — Baseline (ECAPA speaker/accent similarity)

Code + **checkpoint pre-trained** có sẵn trong repo (Baseline 2 fine-tuned).

⏳ **CẦN DATA Track 3** (đang chờ license). Khi có: sửa `DATA_ROOT` rồi bỏ comment các cell `!`.

Điểm dev tham khảo: spk SRCC ~0.45, acc SRCC ~0.44. Format nộp `answer.txt`.

## 0. Config — SỬA khi có data

In [ ]:
DATA_ROOT = '/kaggle/input/<track3-data>'   # << thư mục có sets/ + wav/ (đã copy wav từ _vctk sang _syn/wav)
DEV_CSV = f'{DATA_ROOT}/sets/dev.csv'
T3 = '/kaggle/working/vmc2026-baselines/track3'
OUT_DIR = '/kaggle/working'
CKPT_SPK = f'{T3}/official-egs/spk_sim_adamw_lr1e-3/model_spk_sim_step20000.pt'
CKPT_ACC = f'{T3}/official-egs/acc_sim_adamw_lr1e-3/model_acc_sim_step20000.pt'

## 1. Cài đặt
Repo gốc dùng `uv`; trên Kaggle cài speechbrain + chạy python trực tiếp. Nếu thiếu dep, xem `track3/pyproject.toml`.

In [ ]:
!git clone -q https://github.com/voicemos-challenge/vmc2026-baselines.git /kaggle/working/vmc2026-baselines
!pip install -q speechbrain torchaudio pandas

## 2. Inference (chờ data — bỏ comment khi có DATA_ROOT)
Dùng checkpoint fine-tuned có sẵn. Chạy riêng spk + acc.

In [ ]:
# !cd {T3} && python inference.py --data-root {DATA_ROOT} --csv-path {DEV_CSV} --checkpoint {CKPT_SPK} --out {OUT_DIR}/spk_dev.csv
# !cd {T3} && python inference.py --data-root {DATA_ROOT} --csv-path {DEV_CSV} --checkpoint {CKPT_ACC} --out {OUT_DIR}/acc_dev.csv
print('Bỏ comment 2 dòng trên khi đã có DATA_ROOT')

## 3. Gộp spk + acc → answer.txt
⚠️ Kiểm tra tên cột điểm thực trong output rồi chỉnh `SPK_COL`/`ACC_COL`.

In [ ]:
import pandas as pd
SPK_COL, ACC_COL = 'pred_spk_sim', 'pred_acc_sim'   # << chỉnh theo output thực
KEYS = ['system_id', 'utterance_id', 'wav_a_path', 'wav_b_path']

spk = pd.read_csv(f'{OUT_DIR}/spk_dev.csv').rename(columns={SPK_COL: 'pred_spk_sim'})
acc = pd.read_csv(f'{OUT_DIR}/acc_dev.csv').rename(columns={ACC_COL: 'pred_acc_sim'})
merged = spk.merge(acc[KEYS + ['pred_acc_sim']], on=KEYS, how='outer')
cols = KEYS + ['pred_acc_sim', 'pred_spk_sim']
merged[cols].to_csv(f'{OUT_DIR}/answer.txt', index=False)
print(f'Ghi {len(merged)} dòng → answer.txt')
merged[cols].head()

## 4. Đóng zip nộp

In [ ]:
!cd {OUT_DIR} && zip -j submission_track3.zip answer.txt && unzip -l submission_track3.zip